In [ ]:
!pip install streamlit pyngrok

In [ ]:
!rm -rf ~/.ngrok2/ngrok.yml

In [ ]:
!ngrok config add-authtoken 2wS2Ud8EcmEgJuXQFKL0KQLDMAb_3YXP4mzFVtiGH2DvqAm7r

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load tokenizer and model
with open('/content/model/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

model = load_model('/content/model/lstm_model.h5')

# Label mapping
label_mapping = {'neutral': 1, 'positive': 0, 'negative': 2}
index_to_label = {v: k for k, v in label_mapping.items()}
emoji_map = {'positive': "😊", 'neutral': "😐", 'negative': "😡"}

def clean_text(text):
    import re
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def predict_sentiment(text, max_sequence_length=100):
    cleaned_text = clean_text(text)
    sequence = tokenizer.texts_to_sequences([cleaned_text])
    padded = pad_sequences(sequence, maxlen=max_sequence_length)
    y_pred = model.predict(padded)
    pred_index = np.argmax(y_pred, axis=1)[0]
    label = index_to_label[pred_index]
    confidence = float(np.max(y_pred))
    return f"{label.capitalize()} {emoji_map[label]}", confidence

# Streamlit UI
st.title("🧠 Sentiment Analyzer")
st.write("Enter a sentence and I'll predict the sentiment!")

user_input = st.text_area("Your text here:")

if st.button("Predict"):
    if user_input.strip():
        sentiment, confidence = predict_sentiment(user_input)
        st.success(f"**Prediction:** {sentiment}\n\n**Confidence:** {confidence:.2f}")
    else:
        st.warning("Please enter some text to analyze.")


Overwriting app.py


In [ ]:
from pyngrok import ngrok

# Stop any previous tunnels
ngrok.kill()

# Run Streamlit in background
!streamlit run app.py &>/content/logs.txt &

# Connect ngrok to the port Streamlit runs on (8501)
public_url = ngrok.connect(8501)
print(f"Your app is live at: {public_url}")


Your app is live at: NgrokTunnel: "https://9a27-34-168-61-245.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
!unzip model.zip

Archive:  model.zip
   creating: model/
  inflating: model/lstm_model.h5     
  inflating: model/tokenizer.pkl     
